# 마스킹 검증 전용 (2D SAM2 마스크 -> 3D 동일 색 GLB/HTML)

저장해 둔 세그멘테이션 결과(`{TARGET_FOLDER}_seg_labels.npz/.json`)를 DUSt3R 정렬 좌표계에 매핑해서,
2D 에디터에서 칠한 색과 **동일한 색**으로 3D 포인트클라우드를 칠한다.
가구 배치/바닥면적/거리측정 등 다른 기능은 전부 제외하고 **마스킹이 3D에 제대로 붙었는지 확인**하는 것만 목적.

## 0. 환경 설정

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os, sys, glob
import numpy as np
from PIL import Image

if not os.path.exists("/content/dust3r"):
    !git clone --recursive https://github.com/naver/dust3r.git /content/dust3r

%cd /content/dust3r
!pip install -q -r requirements.txt
!pip install -q open3d trimesh scipy
%cd /content

if "/content/dust3r" not in sys.path:
    sys.path.insert(0, "/content/dust3r")

Cloning into '/content/dust3r'...
remote: Enumerating objects: 611, done.
remote: Total 611 (delta 0), reused 0 (delta 0), pack-reused 611 (from 1)
Receiving objects: 100% (611/611), 756.60 KiB | 1.26 MiB/s, done.
Resolving deltas: 100% (355/355), done.
Submodule 'croco' (https://github.com/naver/croco) registered for path 'croco'
Cloning into '/content/dust3r/croco'...
remote: Enumerating objects: 198, done.        
remote: Counting objects: 100% (87/87), done.        
remote: Compressing objects: 100% (54/54), done.        
remote: Total 198 (delta 54), reused 33 (delta 33), pack-reused 111 (from 1)        
Receiving objects: 100% (198/198), 403.93 KiB | 36.72 MiB/s, done.
Resolving deltas: 100% (94/94), done.
Submodule path 'croco': checked out 'd7de0705845239092414480bd829228723bf20de'
/content/dust3r
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

## 1. 데이터 폴더 지정

In [3]:
# 처리 대상 폴더 지정 (기존 파이프라인과 동일하게 맞출 것)
TARGET_FOLDER = "room06"
CONF_THR = 1.0  # DUSt3R confidence 필터링 기준 (본 파이프라인과 반드시 동일하게)

target_path = f"/content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data/raw_room/{TARGET_FOLDER}"
img_paths = []
for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
    img_paths.extend(glob.glob(os.path.join(target_path, ext)))
img_paths = sorted(img_paths)

print(f"[{TARGET_FOLDER}] 이미지 {len(img_paths)}장")
for p in img_paths:
    print(" ", os.path.basename(p))

images_pil = [Image.open(p).convert("RGB") for p in img_paths]

[room06] 이미지 10장
  img_01.jpg
  img_02.jpg
  img_03.jpg
  img_04.jpg
  img_05.jpg
  img_06.jpg
  img_07.jpg
  img_08.jpg
  img_09.jpg
  img_10.jpg


## 2. 3D 복원 (DUSt3R)

In [4]:
import torch
from dust3r.model import AsymmetricCroCo3DStereo
from dust3r.utils.image import load_images
from dust3r.image_pairs import make_pairs
from dust3r.inference import inference
from dust3r.cloud_opt import global_aligner

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AsymmetricCroCo3DStereo.from_pretrained(
    "naver/DUSt3R_ViTLarge_BaseDecoder_512_dpt"
).to(device)
model.eval()
print(f"DUSt3R 모델 로드 완료 (device: {device})")

Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead


/content/dust3r/dust3r/cloud_opt/base_opt.py:275: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)


config.json:   0%|          | 0.00/450 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

DUSt3R 모델 로드 완료 (device: cuda)


In [5]:
images_dust3r = load_images(img_paths, size=512)
pairs = make_pairs(images_dust3r, scene_graph="complete", prefilter=None, symmetrize=True)
print(f"이미지 쌍 {len(pairs)}개 구성")

with torch.no_grad():
    output = inference(pairs, model, device, batch_size=2)

scene = global_aligner(output, device=device)
scene.compute_global_alignment(niter=300, init="mst")
print("3D 복원 및 정렬 완료")

>> Loading a list of 10 images
 - adding /content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data/raw_room/room06/img_01.jpg with resolution 1440x1920 --> 384x512
 - adding /content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data/raw_room/room06/img_02.jpg with resolution 1440x1920 --> 384x512
 - adding /content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data/raw_room/room06/img_03.jpg with resolution 1440x1920 --> 384x512
 - adding /content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data/raw_room/room06/img_04.jpg with resolution 1440x1920 --> 384x512
 - adding /content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data/raw_room/room06/img_05.jpg with resolution 1440x1920 --> 384x512
 - adding /content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data/raw_room/room06/img_06.jpg with resolution 1440x1920 --> 384x512
 - adding /content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data/raw_room/room06/img_07.jpg with resolution 1440x1920 --> 384x512
 - adding /content/drive/Othercomputer

  0%|          | 0/45 [00:00<?, ?it/s]/content/dust3r/dust3r/inference.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=bool(use_amp)):
/content/dust3r/dust3r/model.py:206: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/content/dust3r/dust3r/inference.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
100%|██████████| 45/45 [00:20<00:00,  2.21it/s]


 init edge (0*,1*) score=np.float64(382.89166259765625)
 init edge (0,9*) score=np.float64(249.53294372558594)
 init edge (2*,1) score=np.float64(191.88235473632812)
 init edge (9,8*) score=np.float64(70.78853607177734)
 init edge (2,3*) score=np.float64(266.15899658203125)
 init edge (3,4*) score=np.float64(236.8054962158203)
 init edge (4,5*) score=np.float64(86.14830780029297)
 init edge (7*,5) score=np.float64(110.87826538085938)
 init edge (7,6*) score=np.float64(263.4967346191406)
 init loss = 0.015211939811706543
Global alignement - optimizing for:
['pw_poses', 'im_depthmaps', 'im_poses', 'im_focals']


  0%|          | 0/300 [00:00<?, ?it/s]/content/dust3r/dust3r/cloud_opt/base_opt.py:366: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  return float(loss), lr
100%|██████████| 300/300 [00:30<00:00,  9.83it/s, lr=1.27413e-06 loss=0.0101352]

3D 복원 및 정렬 완료


## 3. 저장된 세그멘테이션 마스크 불러오기

SAM2/YOLO 에디터는 쓰지 않고, 이미 저장해 둔 `{TARGET_FOLDER}_seg_labels.npz/.json`만 읽는다.
(파일명 basename 기준 매칭이라 경로가 바뀌어도 동작)

In [6]:
import json

SEG_SAVE_BASE = f"/content/drive/Othercomputers/내 노트북/Asac-DL-Team4/02_Data/{TARGET_FOLDER}_seg_labels"

def load_seg_results(base=None):
    """저장된 seg_labels(npz+json) -> img_paths 순서에 맞춘 all_results 리스트
    (predictor/SEG_STATE 등 에디터 의존성 없음)"""
    base = base or SEG_SAVE_BASE
    with open(base + ".json") as f:
        meta = json.load(f)
    arrs = np.load(base + ".npz")
    by_name = {m["filename"]: (mi, m) for mi, m in enumerate(meta["images"])}

    results = []
    for path in img_paths:
        name = os.path.basename(path)
        res = {"masks": [], "boxes": [], "labels": []}
        if name not in by_name:
            print(f"  [경고] 저장본에 없음: {name}")
            results.append(res)
            continue
        mi, m = by_name[name]
        for k, label in enumerate(m["labels"]):
            res["masks"].append(arrs[f"m_{mi}_{k}"].astype(bool))
            res["boxes"].append(np.array(m["boxes"][k], dtype=np.float32))
            res["labels"].append(label)
        print(f"{name}: {len(res['labels'])}개 -> {res['labels']}")
        results.append(res)
    return results

all_results = load_seg_results()
n_total = sum(len(r["labels"]) for r in all_results)
print(f"\n총 {n_total}개 객체 로드 완료")

img_01.jpg: 1개 -> ['bed']
img_02.jpg: 1개 -> ['bed']
img_03.jpg: 2개 -> ['bed', 'chair']
img_04.jpg: 2개 -> ['refrigerator', 'chair']
img_05.jpg: 2개 -> ['refrigerator', 'chair']
img_06.jpg: 0개 -> []
img_07.jpg: 1개 -> ['chair']
img_08.jpg: 3개 -> ['bed', 'laptop', 'chair']
img_09.jpg: 3개 -> ['bed', 'laptop', 'chair']
img_10.jpg: 1개 -> ['bed']

총 16개 객체 로드 완료


## 4. 정렬 좌표 생성 + 마스크 -> 3D 점 라벨 매핑

In [7]:
import trimesh
from scipy.spatial.transform import Rotation

try:
    from dust3r.demo import OPENGL
except ImportError:
    from dust3r.viz import OPENGL

cams2world = scene.get_im_poses().detach().cpu().numpy()
_rot = np.eye(4)
_rot[:3, :3] = Rotation.from_euler("y", np.deg2rad(180)).as_matrix()
GLB_TRANSFORM = np.linalg.inv(cams2world[0] @ OPENGL @ _rot)


def apply_transform(points, T):
    pts_h = np.concatenate([points, np.ones((len(points), 1))], axis=1)
    out = (T @ pts_h.T).T
    return out[:, :3]


def find_floor_rotation(points, up_axis=1, n_iter=2000):
    height = points[:, up_axis]
    floor_candidates = points[height <= np.percentile(height, 35)]
    if len(floor_candidates) < 3:
        return np.eye(4), np.array([0.0, 1.0, 0.0])

    diag = np.linalg.norm(points.max(0) - points.min(0))
    threshold = diag * 0.02

    best_inliers, best_normal = -1, np.array([0.0, 1.0, 0.0])
    for _ in range(n_iter):
        idx = np.random.choice(len(floor_candidates), 3, replace=False)
        p1, p2, p3 = floor_candidates[idx]
        normal = np.cross(p2 - p1, p3 - p1)
        nrm = np.linalg.norm(normal)
        if nrm < 1e-6:
            continue
        normal = normal / nrm
        d = -np.dot(normal, p1)
        dist = np.abs(floor_candidates @ normal + d)
        ninl = int((dist < threshold).sum())
        if ninl > best_inliers:
            best_inliers, best_normal = ninl, normal

    if best_normal[up_axis] < 0:
        best_normal = -best_normal

    target = np.array([0.0, 1.0, 0.0])
    v = np.cross(best_normal, target)
    s = np.linalg.norm(v)
    c = float(np.dot(best_normal, target))
    if s < 1e-8:
        R = np.eye(3)
    else:
        vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
        R = np.eye(3) + vx + vx @ vx * ((1 - c) / (s * s))

    T = np.eye(4)
    T[:3, :3] = R
    return T, best_normal


pointmaps = scene.get_pts3d()
confidences = scene.get_conf()

_all_pts_raw, _all_cols = [], []
for i, pts in enumerate(pointmaps):
    pts_np = pts.detach().cpu().numpy().reshape(-1, 3)
    conf_np = confidences[i].detach().cpu().numpy().flatten()
    mask = conf_np > CONF_THR
    h, w = pts.shape[:2]
    img_resized = np.array(images_pil[i].resize((w, h))).reshape(-1, 3)
    _all_pts_raw.append(pts_np[mask])
    _all_cols.append(img_resized[mask])
_all_pts_raw = np.concatenate(_all_pts_raw, axis=0)
all_cols = np.concatenate(_all_cols, axis=0) / 255.0

_all_pts_glb = apply_transform(_all_pts_raw, GLB_TRANSFORM)
FLOOR_ROT, floor_normal = find_floor_rotation(_all_pts_glb)
ALIGN_TRANSFORM = FLOOR_ROT @ GLB_TRANSFORM

all_pts = apply_transform(_all_pts_raw, ALIGN_TRANSFORM)
print(f"정렬 완료. 전체 포인트 수: {len(all_pts):,}")

정렬 완료. 전체 포인트 수: 1,948,128


In [15]:
import cv2

# 통합 점군에 붙일 라벨 배열. "" = 미분류(배경/바닥/벽)
# * 순수 매핑 확인이 목적이라 oriented-bbox 흡수 등 후처리는 하지 않는다 *
point_labels = np.empty(len(all_pts), dtype=object)
point_labels[:] = ""

_offset = 0
for vi, pts in enumerate(pointmaps):
    h, w = pts.shape[:2]
    conf_np = confidences[vi].detach().cpu().numpy().flatten()
    conf_mask = conf_np > CONF_THR
    n_view = int(conf_mask.sum())

    res = all_results[vi]
    order = sorted(range(len(res["masks"])),
                   key=lambda k: (res["boxes"][k][2]-res["boxes"][k][0]) *
                                 (res["boxes"][k][3]-res["boxes"][k][1]),
                   reverse=True)

    view_label_flat = np.empty(h * w, dtype=object)
    view_label_flat[:] = ""
    for k in order:
        lbl = res["labels"][k]
        m = res["masks"][k].astype(np.uint8)
        m_resized = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)
        view_label_flat[m_resized.flatten()] = lbl

    point_labels[_offset:_offset + n_view] = view_label_flat[conf_mask]
    _offset += n_view

assert _offset == len(all_pts), f"정합 실패: {_offset} != {len(all_pts)}"

uniq, cnts = np.unique(point_labels, return_counts=True)
print("=== 라벨별 3D 점 수 ===")
for u, c in sorted(zip(uniq, cnts), key=lambda x: -x[1]):
    print(f"  {(u if u else '(미분류)'):15s}: {c:>7,}점")

=== 라벨별 3D 점 수 ===
  (미분류)          : 1,606,222점
  bed            : 248,885점
  refrigerator   :  54,615점
  chair          :  34,253점
  laptop         :   4,153점


## 5. 마스킹 검증 GLB/HTML (2D 마스크와 동일한 색)

라벨 있는 점만 `label_color()`(2D 에디터와 동일한 고정 색상표)로 칠하고,
나머지(배경/바닥/벽)는 원본 사진색을 어둡게 깔아 구조 참고용으로 남긴다.

In [19]:
# =========================================================
# [검증] verify_mesh 마스킹 강조 O vs OFF 비교 (한 뷰어에서 버튼 토글)
#   재질/조명은 둘 다 완전히 동일한 코드 -> 차이가 보이면 순수 마스킹 오버레이 때문.
#   ② 강조 O : dim=0.95, alpha=0.55 (배경 살짝 어둡게 + 마스크에 라벨색 블렌딩)
#   ③ 강조 OFF: dim=1.0, alpha=0.0  (오버레이 없이 원본 사진 색 그대로)
#   전제: scene, all_results, confidences, ALIGN_TRANSFORM, CONF_THR, images_pil 존재
# =========================================================
import base64, json as _json, numpy as np, cv2, trimesh

try:
    from dust3r.demo import pts3d_to_trimesh, cat_meshes
except ImportError:
    from dust3r.viz import pts3d_to_trimesh, cat_meshes
from dust3r.utils.device import to_numpy

LABEL_COLORS = {
    "bed": (255, 59, 59),
    "desk": (59, 179, 255),
    "wardrob": (59, 255, 107),
    "refrigerator": (255, 200, 59),
    "couch": (200, 59, 255),
}
_EXTRA_COLORS = [(120, 220, 220), (220, 120, 180), (180, 220, 120), (140, 140, 255)]

def label_color(label):
    if label in LABEL_COLORS:
        return LABEL_COLORS[label]
    return _EXTRA_COLORS[abs(hash(label)) % len(_EXTRA_COLORS)]


def build_verify_mesh(all_results, conf_thr=CONF_THR, dim=0.95, alpha=0.55):
    """뷰별로 원본 사진 위에 마스크 부분만 라벨색을 alpha 블렌딩한다.
    dim=1.0, alpha=0.0 이면 오버레이 없이 순수 사진 색."""
    rgbimg = scene.imgs
    pts3d_np = to_numpy(scene.get_pts3d())

    pieces = []
    for i, img in enumerate(rgbimg):
        H, W = img.shape[:2]
        conf_np = confidences[i].detach().cpu().numpy()
        valid = conf_np > conf_thr

        res = all_results[i]
        order = sorted(range(len(res["masks"])),
                       key=lambda k: (res["boxes"][k][2]-res["boxes"][k][0]) *
                                     (res["boxes"][k][3]-res["boxes"][k][1]),
                       reverse=True)
        label_flat = np.full((H, W), "", dtype=object)
        for k in order:
            lbl = res["labels"][k]
            m = cv2.resize(res["masks"][k].astype(np.uint8), (W, H),
                           interpolation=cv2.INTER_NEAREST).astype(bool)
            label_flat[m] = lbl

        color_img = img.astype(np.float32) * dim
        if alpha > 0:
            for lbl in np.unique(label_flat):
                if lbl == "":
                    continue
                c = np.array(label_color(lbl), dtype=np.float32) / 255.0
                m = label_flat == lbl
                color_img[m] = img[m].astype(np.float32) * (1 - alpha) + c * alpha

        piece = pts3d_to_trimesh(color_img, pts3d_np[i], valid)
        if len(piece["faces"]):
            pieces.append(piece)

    mesh = trimesh.Trimesh(**cat_meshes(pieces))
    mesh.apply_transform(ALIGN_TRANSFORM)
    mesh.remove_unreferenced_vertices()
    return mesh


def _mesh_payload(mesh):
    v = mesh.vertices.astype(np.float32)
    f = mesh.faces.astype(np.uint32)
    c = (mesh.visual.vertex_colors[:, :3].astype(np.float32) / 255.0)
    return {
        "v_b64": base64.b64encode(v.tobytes()).decode("ascii"),
        "f_b64": base64.b64encode(f.tobytes()).decode("ascii"),
        "c_b64": base64.b64encode(c.tobytes()).decode("ascii"),
        "n_v": len(v), "n_f": len(f),
    }

# ② 마스킹 강조 O
verify_mesh_on = build_verify_mesh(all_results, dim=0.95, alpha=0.55)
_p_on = _mesh_payload(verify_mesh_on)
print(f"② 강조 O  : 정점 {_p_on['n_v']:,} / 면 {_p_on['n_f']:,}")

# ③ 마스킹 강조 OFF
verify_mesh_off = build_verify_mesh(all_results, dim=1.0, alpha=0.0)
_p_off = _mesh_payload(verify_mesh_off)
print(f"③ 강조 OFF: 정점 {_p_off['n_v']:,} / 면 {_p_off['n_f']:,}")

_PAYLOADS = {"on": _p_on, "off": _p_off}

_HTML = r"""
<!DOCTYPE html><html><head><meta charset="utf-8"><style>
  body{margin:0;overflow:hidden;font-family:sans-serif;background:#111111}
  #ui{position:absolute;top:12px;left:12px;background:rgba(20,20,30,.85);
      color:#eee;padding:14px 16px;border-radius:10px;min-width:230px}
  #ui h3{margin:0 0 8px;font-size:13px;color:#9a9ad0}
  .btnrow{display:flex;flex-direction:column;gap:6px}
  .btnrow button{background:#2a2a3a;color:#eee;border:1px solid #444;
      padding:8px 10px;border-radius:6px;cursor:pointer;font-size:12px;text-align:left}
  .btnrow button:hover{border-color:#8b8bd0}
  .btnrow button.active{background:#4338ca;border-color:#818cf8}
  #note{margin-top:10px;padding-top:8px;border-top:1px solid #333;font-size:11px;color:#999;line-height:1.5}
</style></head><body>
<div id="ui">
  <h3>마스킹 강조 비교</h3>
  <div class="btnrow">
    <button id="btn-on" class="active">② 마스킹 강조 O</button>
    <button id="btn-off">③ 마스킹 강조 OFF (원본색)</button>
  </div>
  <div id="note">재질·조명은 두 버튼 동일 코드.<br>다르게 보이면 마스킹 오버레이 때문.</div>
</div>
<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/controls/OrbitControls.js"></script>
<script>
const PAYLOADS = __PAYLOADS__;

function b64buf(b){const s=atob(b),n=s.length,a=new Uint8Array(n);
  for(let i=0;i<n;i++)a[i]=s.charCodeAt(i);return a.buffer;}

const scene=new THREE.Scene();
scene.background=new THREE.Color(0x111111);
const camera=new THREE.PerspectiveCamera(55,innerWidth/innerHeight,0.001,1000);
const renderer=new THREE.WebGLRenderer({antialias:true});
renderer.setSize(innerWidth,innerHeight);document.body.appendChild(renderer.domElement);
const controls=new THREE.OrbitControls(camera,renderer.domElement);

// 동일한 조명
scene.add(new THREE.AmbientLight(0xffffff,0.95));
const dl=new THREE.DirectionalLight(0xffffff,0.5);dl.position.set(1,1,1);scene.add(dl);

const meshObjs={};
let framed=false;

function buildMesh(key){
  const p=PAYLOADS[key];
  const posArr=new Float32Array(b64buf(p.v_b64));
  const colArr=new Float32Array(b64buf(p.c_b64));
  const idxArr=new Uint32Array(b64buf(p.f_b64));
  const geo=new THREE.BufferGeometry();
  geo.setAttribute("position", new THREE.BufferAttribute(posArr,3));
  geo.setAttribute("color", new THREE.BufferAttribute(colArr,3));
  geo.setIndex(new THREE.BufferAttribute(idxArr,1));
  geo.computeVertexNormals();
  // 동일한 재질
  const mat=new THREE.MeshStandardMaterial({
    vertexColors:true, side:THREE.DoubleSide, metalness:0.0, roughness:1.0
  });
  const m=new THREE.Mesh(geo, mat);
  m.visible=false;
  scene.add(m);
  meshObjs[key]=m;
  if(!framed){
    geo.computeBoundingSphere();
    const c=geo.boundingSphere.center, r=geo.boundingSphere.radius||1;
    camera.position.set(c.x+r*1.2,c.y+r*0.8,c.z+r*1.2);
    controls.target.copy(c);controls.update();
    framed=true;
  }
}
["on","off"].forEach(buildMesh);

function show(key){
  Object.keys(meshObjs).forEach(k=>meshObjs[k].visible=(k===key));
  ["on","off"].forEach(k=>{
    document.getElementById("btn-"+k).classList.toggle("active", k===key);
  });
}
document.getElementById("btn-on").onclick=()=>show("on");
document.getElementById("btn-off").onclick=()=>show("off");
show("on");

addEventListener("resize",()=>{camera.aspect=innerWidth/innerHeight;
  camera.updateProjectionMatrix();renderer.setSize(innerWidth,innerHeight);});
(function loop(){requestAnimationFrame(loop);controls.update();
  renderer.render(scene,camera);})();
</script></body></html>
"""
_HTML = _HTML.replace("__PAYLOADS__", _json.dumps(_PAYLOADS))

_name = globals().get("SAFE_NAME", TARGET_FOLDER)
_out = f"/content/{_name}_mask_emphasis_compare.html"
with open(_out, "w") as f:
    f.write(_HTML)
print("저장:", _out)

from google.colab import files
files.download(_out)

② 강조 O  : 정점 1,948,123 / 면 7,747,616
③ 강조 OFF: 정점 1,948,123 / 면 7,747,616
저장: /content/room06_mask_emphasis_compare.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>